# Porter: Delivery-Time Estimation with a Neural Network (Regression)

**Case study:** Porter NN Regression · **Course:** Scaler DSML, Neural Networks
**Goal:** predict how many minutes a delivery will take, using what was ordered, from where, and how many delivery partners are available. Then compare a Random Forest baseline against a neural network.

This notebook follows the **Evaluation Criteria (100 points)** of the problem brief. Each section below is one scoring block, in the same order as the brief.

| # | Section (evaluation criterion) | Points | Status in this version |
|---|---|---|---|
| 1 | Defining the problem statement, importing the data and data structure analysis | 10 | **Complete** |
| 2 | Data preprocessing and feature engineering | 30 | **Complete** |
| 3 | Data visualization and cleaning (outliers) | 20 | **Complete** |
| 4 | Model training with Random Forest | 10 | **Complete** |
| 5 | Regression with neural networks | 30 | **Complete** |

**Questions from the brief and where they are answered**

| Question in the brief | Answered in |
|---|---|
| Define the problem statement and where it (and modifications of it) can be used | Section 1.1 |
| List 3 functions the pandas datetime provides, with one line each | Section 1.6 |
| Short note on datetime, timedelta and timespan (period) | Section 1.7 |
| Why check for outliers? Name 3 outlier removal methods | Section 3 |
| Other classical ML methods besides random forest for regression | Section 4 |
| Why scaling for neural networks? Choice of optimizer? Activation function? Why NNs do well on large data? | Section 5 |

---
# Section 1: Problem statement, data import and data structure analysis (10 points)

**What this section covers**
1. Define the business problem, the ML framing, and where the idea can be reused.
2. Import the dataset into a pandas DataFrame.
3. Inspect its structure: shape, columns, data types, missing values, duplicates, value ranges.
4. Note which columns need type conversion or cleaning later, without changing the data yet.
5. Answer the two pandas datetime questions of the brief.

## 1.1 Problem statement (brief question 1)

**Context.** Porter is India's largest marketplace for intra-city logistics. It supports 1,50,000+ driver-partners and has served 5+ million customers. Porter also works with a wide range of restaurants whose food is delivered directly to people.

**Business problem.** At the moment a customer places an order, Porter needs to tell them *how long it will take to arrive*. That estimate should depend on:
- **what** is being ordered (number of items, price, number of distinct items),
- **where** it is ordered from (market, restaurant category, how the order was placed),
- **who can deliver it** right now (delivery partners on duty, busy partners, orders already waiting).

**Why it matters**

| Who | What the estimate drives |
|---|---|
| Customer | Whether they order at all, and whether they trust Porter. An optimistic promise that is missed hurts trust. A pessimistic promise loses orders. |
| Driver-partner | Fair planning of trips and earnings. |
| Operations team | Dispatch and staffing decisions. Early warning when a market is short of partners. |

**ML formulation**

| Item | Choice |
|---|---|
| Task type | Supervised **regression** (the answer is a number, not a class) |
| Target | `time_taken_mins` = `actual_delivery_time` − `created_at`, converted to minutes (created in Section 2) |
| Features | Only information known when the order is placed. `actual_delivery_time` is the source of the target and must **not** be used as a feature (that would be data leakage). |
| Metrics | MAE (average error in minutes), RMSE (penalises big misses), R², plus a business view such as "share of orders predicted within ±5 or ±10 minutes" |
| Models | Random Forest baseline, then a PyTorch neural network, compared on the same test set |

**Where this idea (and modifications of it) can be used**

*Inside Porter*
- Delivery ETA for goods and parcel trips, not only food.
- Pickup ETA: how long until a driver-partner reaches the restaurant or sender.
- Choosing delivery promises or slots, and setting prices when partners are scarce.
- Deciding how many partners to keep on shift in each market and hour.

*Outside Porter*
- Food delivery and quick-commerce apps (the same features: items, price, restaurant, rider supply).
- E-commerce last-mile delivery and courier or freight companies.
- Ride-hailing arrival and trip-duration estimates.
- Field-service appointment windows and ambulance response times.

*Useful modifications*
- Predict a range (for example the 10th and 90th percentile) instead of one number, so the app can show "35 to 50 min".
- Turn it into classification: "will this order be late against its promise?" (the course's QuickCart example does this).
- Multi-task model: predict delivery time and delivery cost together from the same features.
- Use live features such as traffic and weather, and retrain regularly because conditions drift.

*One other scenario in detail: ride-hailing arrival time*

| | Porter delivery time | Ride-hailing arrival time |
|---|---|---|
| Predicted | Minutes from order to delivery | Minutes from booking to pickup |
| Known at request time | Items, price, restaurant, partners on duty and busy, orders waiting | Pickup distance, drivers nearby and busy, requests waiting, time of day |
| Problem type | Regression | Regression |
| Why it transfers | Supply (partners) versus demand (orders) drives the wait | Supply (drivers) versus demand (requests) drives the wait |

## 1.2 Setup: libraries and settings

In [ ]:
import warnings
from pathlib import Path
import gdown

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

SEED = 42
np.random.seed(SEED)

BLUE, ORANGE, AQUA, YELLOW, GRAY = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#8a8a85"
sns.set_theme(style="whitegrid", palette=[BLUE, ORANGE, AQUA, YELLOW])
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 100})

print("pandas", pd.__version__, "| numpy", np.__version__)

## 1.3 Import the data

The brief links the dataset on Google Drive. A copy is stored in `data/porter_delivery.csv` next to this notebook. The cell below uses the local copy and downloads it from Drive (with `gdown`) only if it is missing. Each row is **one unique delivery**.

In [ ]:
DATA_PATH = Path("data/porter_delivery.csv")
DRIVE_ID = "1kkgBGldeswHBgVKEJiqqG7VZvqZNMc-5"          # file id from the problem brief

if not DATA_PATH.exists():
    DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    print("Local copy not found, downloading from Google Drive ...")
    gdown.download(id=DRIVE_ID, output=str(DATA_PATH))

df = pd.read_csv(DATA_PATH)
print(f"Loaded {DATA_PATH}  ->  {df.shape[0]:,} rows x {df.shape[1]} columns")

## 1.4 Data structure analysis

### 1.4.1 First look at the rows

In [ ]:
df.head()

In [ ]:
df.tail(3)

### 1.4.2 Shape, memory, column names and data types

In [ ]:
rows, cols = df.shape
print(f"Rows (deliveries): {rows:,}")
print(f"Columns          : {cols}")
print(f"Memory used      : {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
print("\nColumn names:")
for i, c in enumerate(df.columns, 1):
    print(f"  {i:>2}. {c}")

In [ ]:
df.info()

### 1.4.3 One summary table: type, missing values, unique values, example

In [ ]:
summary = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "non_null": df.notna().sum(),
    "nulls": df.isna().sum(),
    "null_%": (df.isna().mean() * 100).round(2),
    "unique": df.nunique(),
    "example": df.iloc[0],
})
summary

### 1.4.4 Missing values and duplicate rows

In [ ]:
total_nulls = int(df.isna().sum().sum())
dup_rows = int(df.duplicated().sum())
print(f"Total missing cells : {total_nulls}")
print(f"Duplicate rows      : {dup_rows}")
print("Columns with nulls  :", list(df.columns[df.isna().any()]) or "none")

### 1.4.5 Statistical summary of the numeric columns

In [ ]:
df.describe().T

`describe()` skips the text columns. The cell below summarises them: count, number of unique values, most frequent value and its frequency.

In [ ]:
df.describe(exclude="number")

### 1.4.6 The two timestamp columns (currently plain text)
`created_at` and `actual_delivery_time` are stored as text (`str` / object). The check below parses them **temporarily** to understand their range. The real conversion belongs to Section 2.

In [ ]:
print("dtype created_at           :", df["created_at"].dtype)
print("dtype actual_delivery_time :", df["actual_delivery_time"].dtype)

created = pd.to_datetime(df["created_at"])
delivered = pd.to_datetime(df["actual_delivery_time"])

print("\nOrders placed from :", created.min())
print("Orders placed to   :", created.max())
print("Period covered     :", (created.max() - created.min()).days, "days")
print("Any delivery before order?   ", bool((delivered < created).any()))

### 1.4.6b Variable types and conversions needed

| Column | Stored as now | What it really is | Conversion needed |
|---|---|---|---|
| `created_at` | text | Date and time of order | Convert to datetime (Section 2) |
| `actual_delivery_time` | text | Date and time of delivery | Convert to datetime, build the target, then drop (Section 2) |
| `market_id`, `order_protocol` | `float64` | Category labels (6 and 7 values) | Treat as categories |
| `store_primary_category` | `int64` | Category label (73 values) | Encode as integer codes (Section 2) |
| Counts: `total_items`, `num_distinct_items`, partner and order counts | `int64` / `float64` | Counts | None, but check negatives and extremes |
| Prices: `subtotal`, `min_item_price`, `max_item_price` | `int64` | Prices | None, but check zeros, negatives and extremes |
| `estimated_store_to_consumer_driving_duration` | `float64` | Duration in seconds (not described in the brief) | Not used (Section 2.4) |

In [ ]:
df.nunique().to_frame("distinct_values")

### 1.4.7 Columns that look numeric but are really categories (IDs)
`market_id`, `order_protocol` and `store_primary_category` are stored as numbers, but the number is only a label. "Market 4" is not twice "Market 2". They need categorical treatment later. `market_id` and `order_protocol` are even stored as decimals (`1.0`, `2.0`), another sign that they are labels and not quantities.

In [ ]:
for col in ["market_id", "order_protocol"]:
    print(f"--- {col} ({df[col].nunique()} distinct values) ---")
    print(df[col].value_counts().sort_index().to_string(), "\n")

cat = df["store_primary_category"]
print(f"--- store_primary_category ({cat.nunique()} distinct values), 10 most common ---")
print(cat.value_counts().head(10).to_string())

Market 2 has the most orders (53,469) and market 6 the fewest (638). Protocols 1, 3 and 5 carry most orders, protocol 7 only 19. Restaurant category 4 is the most common (18,183 orders).

### 1.4.8 A peek at the future target (temporary, not saved)
The target is the time between the two timestamps. Seeing its range now tells us what a sensible prediction error looks like. It is computed on a temporary series only.

In [ ]:
peek_minutes = (delivered - created).dt.total_seconds() / 60
peek_minutes.describe().round(2)

### 1.4.9 Data-quality flags to handle later
These checks do not change the data. They list values that look wrong or suspicious so Sections 2 and 3 can deal with them on purpose.

In [ ]:
flags = {
    "min_item_price < 0": (df["min_item_price"] < 0).sum(),
    "total_onshift_dashers < 0": (df["total_onshift_dashers"] < 0).sum(),
    "total_busy_dashers < 0": (df["total_busy_dashers"] < 0).sum(),
    "total_outstanding_orders < 0": (df["total_outstanding_orders"] < 0).sum(),
    "subtotal == 0": (df["subtotal"] == 0).sum(),
    "min_item_price > max_item_price": (df["min_item_price"] > df["max_item_price"]).sum(),
    "busy dashers > onshift dashers": (df["total_busy_dashers"] > df["total_onshift_dashers"]).sum(),
    "driving duration == 0": (df["estimated_store_to_consumer_driving_duration"] == 0).sum(),
}
flag_table = pd.DataFrame({"rows": pd.Series(flags).astype(int)})
flag_table["share_of_data_%"] = (flag_table["rows"] / len(df) * 100).round(2)
flag_table

In [ ]:
extremes = pd.DataFrame({
    "median": df[["total_items", "subtotal", "max_item_price", "total_outstanding_orders"]].median(),
    "99th pct": df[["total_items", "subtotal", "max_item_price", "total_outstanding_orders"]].quantile(0.99),
    "max": df[["total_items", "subtotal", "max_item_price", "total_outstanding_orders"]].max(),
})
extremes

## 1.5 Data dictionary: brief versus the file we actually have

The brief's data dictionary and the downloaded file are not identical. The table maps one to the other, so no column is silently misunderstood.

| Column in the file | Meaning | Type of variable | Note |
|---|---|---|---|
| `market_id` | Market where the restaurant lies | Categorical (ID) | Number is a label only |
| `created_at` | Time the order was placed | Timestamp | Text now, needs conversion |
| `actual_delivery_time` | Time the order was delivered | Timestamp | Source of the target, **not a feature** |
| `store_primary_category` | Restaurant category | Categorical (ID) | Many distinct values |
| `order_protocol` | How the order was placed (Porter, call, pre-booked, third party, ...) | Categorical (ID) | Number is a label only |
| `total_items` | Number of items in the order | Numeric count | The brief's text "total_items subtotal" is two columns run together |
| `subtotal` | Final price of the order | Numeric | |
| `num_distinct_items` | Number of different items | Numeric count | |
| `min_item_price` | Price of the cheapest item | Numeric | |
| `max_item_price` | Price of the costliest item | Numeric | |
| `total_onshift_dashers` | Delivery partners on duty when the order was placed | Numeric count | Brief calls it `total_onshift_partners` |
| `total_busy_dashers` | Partners busy with other tasks | Numeric count | Brief calls it `total_busy_partners` |
| `total_outstanding_orders` | Orders waiting to be fulfilled | Numeric count | |
| `estimated_store_to_consumer_driving_duration` | Estimated driving time from store to customer, in seconds | Numeric | **Present in the file but not described in the brief. Not used** as a feature (Section 2.4). |

In [ ]:
expected = [
    "market_id", "created_at", "actual_delivery_time", "store_primary_category",
    "order_protocol", "total_items", "subtotal", "num_distinct_items",
    "min_item_price", "max_item_price", "total_onshift_dashers",
    "total_busy_dashers", "total_outstanding_orders",
    "estimated_store_to_consumer_driving_duration",
]
missing_cols = [c for c in expected if c not in df.columns]
extra_cols = [c for c in df.columns if c not in expected]
print("Expected columns missing from file:", missing_cols or "none")
print("Columns in file not in the list    :", extra_cols or "none")

## 1.6 Question: three functions the pandas datetime tools provide

| Function / accessor | One-line explanation |
|---|---|
| `pd.to_datetime()` | Parses text (or numbers) into proper datetime values so dates can be compared, sorted and subtracted. |
| `Series.dt.<part>` (for example `.dt.hour`, `.dt.day_of_week`, `.dt.month`) | Pulls one component, such as hour of day or weekday, out of a datetime column. |
| Subtracting two datetimes, then `.dt.total_seconds()` | Gives the time between two timestamps as a `Timedelta`, which `total_seconds()` turns into a plain number (divide by 60 for minutes). |

Other useful ones: `pd.date_range()` builds a series of dates, `Series.dt.floor()` rounds timestamps down to the hour or day, and `DataFrame.resample()` groups time-indexed data into periods.

The cell below shows each one on the first 5 real rows. It works on a copy, so `df` is unchanged.

In [ ]:
demo = df[["created_at", "actual_delivery_time"]].head(5).copy()

# 1) pd.to_datetime: text -> datetime
demo["created_dt"] = pd.to_datetime(demo["created_at"])
demo["delivered_dt"] = pd.to_datetime(demo["actual_delivery_time"])

# 2) .dt accessor: pull parts out of a datetime
demo["hour"] = demo["created_dt"].dt.hour
demo["day_of_week"] = demo["created_dt"].dt.day_of_week          # Monday = 0
demo["day_name"] = demo["created_dt"].dt.day_name()

# 3) subtract two datetimes -> Timedelta -> minutes
demo["time_taken"] = demo["delivered_dt"] - demo["created_dt"]
demo["minutes"] = demo["time_taken"].dt.total_seconds() / 60

demo[["created_dt", "hour", "day_of_week", "day_name", "time_taken", "minutes"]]

In [ ]:
print("created_dt   dtype:", demo["created_dt"].dtype)
print("time_taken   dtype:", demo["time_taken"].dtype)
print("minutes      dtype:", demo["minutes"].dtype)
print("original df is unchanged ->", df["created_at"].dtype)

### What the `.dt` accessor provides by default

A datetime column comes with many ready-made parts and functions under `.dt`. `dir()` lists them, and the table shows some of the most useful ones on the first order.

In [ ]:
dt_tools = [a for a in dir(created.dt) if not a.startswith("_")]
print("Number of attributes and functions in .dt:", len(dt_tools))

first_order = created.iloc[:1]
pd.Series({
    "year": first_order.dt.year.iloc[0],
    "month": first_order.dt.month.iloc[0],
    "day": first_order.dt.day.iloc[0],
    "hour": first_order.dt.hour.iloc[0],
    "minute": first_order.dt.minute.iloc[0],
    "day_of_week (Monday = 0)": first_order.dt.day_of_week.iloc[0],
    "day_name()": first_order.dt.day_name().iloc[0],
    "quarter": first_order.dt.quarter.iloc[0],
    "is_month_end": first_order.dt.is_month_end.iloc[0],
    "floor('h')": first_order.dt.floor("h").iloc[0],
    "strftime('%d %b %Y')": first_order.dt.strftime("%d %b %Y").iloc[0],
}, name="value for the first order")

## 1.7 Question: short note on `datetime`, `timedelta` and `Period` (timespan)

| Concept | What it represents | pandas type | Example from this data |
|---|---|---|---|
| **datetime** | A single **point in time** | `Timestamp` (stored as `datetime64`) | The moment an order was placed: `2015-02-06 22:24:17` |
| **timedelta** | A **duration**, the length of time between two points. It has no start or end of its own. | `Timedelta` (stored as `timedelta64`) | Delivery time = delivered minus placed = `0 days 00:47:00` |
| **Period** (timespan) | A **fixed span on the calendar**, with a start and an end | `Period` | The whole month `2015-02`, or the whole day `2015-02-06` |

How they combine:
- `datetime − datetime` gives a `timedelta`.
- `datetime + timedelta` gives a new `datetime`.
- A `datetime` can be placed into a `Period` (`ts.to_period("M")`), and a `Period` can report its `start_time` and `end_time`.

Our target, delivery time, is a **timedelta**. We will convert it to a plain number of minutes because machine learning models need numbers.

In [ ]:
t_placed = pd.Timestamp(df.loc[0, "created_at"])
t_delivered = pd.Timestamp(df.loc[0, "actual_delivery_time"])

# datetime: a point in time
print("datetime  :", t_placed, "| type:", type(t_placed).__name__)

# timedelta: a duration
duration = t_delivered - t_placed
print("timedelta :", duration, "| type:", type(duration).__name__,
      "| in minutes:", duration.total_seconds() / 60)

# datetime + timedelta -> datetime
print("placed + 30 min       :", t_placed + pd.Timedelta(minutes=30))

# Period: a span on the calendar
month = t_placed.to_period("M")
day = t_placed.to_period("D")
print("Period (month)        :", month, "| from", month.start_time, "to", month.end_time)
print("Period (day)          :", day, "| from", day.start_time, "to", day.end_time)
print("Is the order inside that month?", t_placed >= month.start_time and t_placed <= month.end_time)

## 1.8 Section 1 summary

In [ ]:
invalid_mask = (
    (df["min_item_price"] < 0) | (df["total_onshift_dashers"] < 0) | (df["total_busy_dashers"] < 0)
    | (df["total_outstanding_orders"] < 0) | (df["subtotal"] == 0)
    | (df["min_item_price"] > df["max_item_price"])
)
print("SECTION 1 SUMMARY")
print("-" * 60)
print(f"Dataset              : {df.shape[0]:,} deliveries x {df.shape[1]} columns")
print(f"Missing cells        : {total_nulls}")
print(f"Duplicate rows       : {dup_rows}")
print(f"Period covered       : {created.min().date()} to {created.max().date()}")
print(f"Text columns to convert to datetime : {[c for c in df.columns if 'time' in c or c.endswith('_at')]}")
print(f"ID-like numeric columns             : ['market_id', 'order_protocol', 'store_primary_category']")
print(f"Rows with at least one invalid value (negatives, zero subtotal, min>max price): {int(invalid_mask.sum()):,} ({invalid_mask.mean()*100:.2f}%)")
print(f"Target (preview)     : delivery time between {peek_minutes.min():.0f} and {peek_minutes.max():.0f} min, mean {peek_minutes.mean():.1f}")

### Checklist against the brief

| Task / question in the brief | Where answered | Done |
|---|---|---|
| Task 1.1 and Question 1: business problem, one other scenario, uses and modifications | 1.1 | Yes |
| Task 1.2: import the dataset into a DataFrame | 1.3 | Yes |
| Task 1.3: shape, column names, data types, columns needing conversion | 1.4.2, 1.4.3, 1.4.6b, 1.5 | Yes |
| Structure and characteristics (nulls, duplicates, ranges, categories, quality flags) | 1.4.4 to 1.4.9 | Yes |
| Task 1.4 and Question 2: three pandas datetime functions | 1.6 | Yes |
| Task 1.5 and Question 3: datetime, timedelta, Period | 1.7 | Yes |

### Key observations from Section 1

| # | Observation | What it means for the next sections |
|---|---|---|
| 1 | **175,777 deliveries, 14 columns.** No missing cells and no duplicate rows. | Null handling in Section 2 will confirm this and document the rule (drop if rare, otherwise impute). |
| 2 | Orders were placed over **27 days** (21 Jan to 18 Feb 2015). | A short window: day-of-week patterns can be learned, but monthly or seasonal patterns cannot. |
| 3 | `created_at` and `actual_delivery_time` are **text**. | Convert to datetime first (Section 2.1), then build the target and the time features. |
| 4 | `market_id`, `order_protocol` and `store_primary_category` are numeric **labels** (6, 7 and 73 distinct values). | Treat as categories. `store_primary_category` has many values, so its encoding needs care. |
| 5 | The target preview runs from **32 to 110 minutes**, mean 46.2, median 45, standard deviation 9.3. | A model that always predicts about 46 minutes is off by roughly 9 minutes on average, so any useful model must beat that. |
| 6 | **940 rows (0.53%)** have an impossible value: negative prices or partner counts, zero subtotal, or minimum price above maximum price. | Remove or fix them during cleaning (Sections 2 and 3). |
| 7 | In **22.35%** of rows, busy partners exceed on-shift partners. | Too common to be a typo, so do not delete these rows. Keep the columns and let the models use the information, or build a ratio feature with care. |
| 8 | Heavy right tails: `total_items` has a median of 3 but a maximum of 411. `subtotal` has a 99th percentile of 9,460 and a maximum of 26,800. | These are the outlier candidates for Section 3. |
| 9 | The file has one column the brief does not describe: `estimated_store_to_consumer_driving_duration`. | We follow the brief and do not use it as a feature. |

<a id="sec2"></a>

# Section 2: Data Preprocessing and Feature Engineering (30 points)

In Section 1 we understood the data. Now we prepare it for modelling, following the steps of the problem brief: convert the timestamps, create the target (time taken in minutes), get the hour of the day and the day of the week, clean the data, handle null values and encode the categorical columns.

## 2.1 Convert Timestamps to Datetime

`pd.to_datetime` turns text into proper datetime values. First we keep an untouched copy of the data in `df_raw`, so that we can compare before and after later.

In [ ]:
df_raw = df.copy()

df["created_at"] = pd.to_datetime(df["created_at"])
df["actual_delivery_time"] = pd.to_datetime(df["actual_delivery_time"])

df.info()

## 2.2 Create the Target: Time Taken in Minutes

Subtracting two datetime columns gives a `Timedelta`. `.dt.total_seconds()` turns it into seconds, and dividing by 60 gives the delivery time in minutes, a plain number a model can learn.

In [ ]:
df["time_taken_mins"] = (df["actual_delivery_time"] - df["created_at"]).dt.total_seconds() / 60

df[["created_at", "actual_delivery_time", "time_taken_mins"]].head()

In [ ]:
df["time_taken_mins"].describe()

A histogram shows the shape of the new target column.

In [ ]:
plt.figure(figsize=(9, 4))
sns.histplot(df["time_taken_mins"], bins=range(30, 115, 2), kde=True, color=BLUE)
plt.axvline(df["time_taken_mins"].mean(), color=ORANGE, linestyle="--", linewidth=1.5, label=f"mean = {df['time_taken_mins'].mean():.1f}")
plt.axvline(df["time_taken_mins"].median(), color="#52514e", linestyle=":", linewidth=1.5, label=f"median = {df['time_taken_mins'].median():.0f}")
plt.title("Delivery time in minutes")
plt.xlabel("minutes from order to delivery")
plt.ylabel("number of orders")
plt.legend()
plt.show()

## 2.3 Hour of the Day and Day of the Week

The `.dt` accessor pulls the hour and the weekday out of the order time. Monday is 0 and Sunday is 6.

In [ ]:
df["hour"] = df["created_at"].dt.hour
df["day_of_week"] = df["created_at"].dt.day_of_week

df[["created_at", "hour", "day_of_week"]].head()

## 2.4 Drop Columns That Are No Longer Needed

- `created_at` is now represented by `hour` and `day_of_week`.
- `actual_delivery_time` is the source of the target, so keeping it would give the answer to the model (data leakage).
- `estimated_store_to_consumer_driving_duration` is in the file, but it is not part of the data dictionary of the problem brief. We follow the brief and use only the columns it describes.

In [ ]:
df = df.drop(columns=["created_at", "actual_delivery_time", "estimated_store_to_consumer_driving_duration"])

df.columns.tolist()

## 2.5 Data Cleaning: Impossible Values

In Section 1 we found values that cannot be real: negative prices, negative partner counts, a zero subtotal, and a cheapest item that costs more than the costliest. First we count how many rows break each rule.

In [ ]:
rules = {
    "min_item_price < 0": df["min_item_price"] < 0,
    "total_onshift_dashers < 0": df["total_onshift_dashers"] < 0,
    "total_busy_dashers < 0": df["total_busy_dashers"] < 0,
    "total_outstanding_orders < 0": df["total_outstanding_orders"] < 0,
    "subtotal == 0": df["subtotal"] == 0,
    "min_item_price > max_item_price": df["min_item_price"] > df["max_item_price"],
}

rule_counts = pd.Series({name: int(mask.sum()) for name, mask in rules.items()})
rule_counts

In [ ]:
plt.figure(figsize=(9, 3.5))
ax = sns.barplot(x=rule_counts.values, y=rule_counts.index, color=BLUE)
ax.bar_label(ax.containers[0], fmt="{:,.0f}", fontsize=9)
plt.title("Rows that break each rule (one row can break more than one)")
plt.xlabel("number of rows")
plt.show()

These rows are a tiny share of the data, so we remove them. `pd.concat(...).any(axis=1)` marks a row if it breaks at least one rule.

The rows where busy partners exceed on-duty partners are **not** removed. There are far too many of them to be mistakes (Section 1.3).

In [ ]:
bad_rows = pd.concat(rules.values(), axis=1).any(axis=1)
print("Rows removed:", bad_rows.sum(), f"({bad_rows.mean() * 100:.2f}% of the data)")

df = df[~bad_rows].reset_index(drop=True)
print("Shape after cleaning:", df.shape)

## 2.6 Null Values

`df.isnull().sum()` counts the null values in every column. If only a tiny share of rows had nulls we would drop them, and otherwise we would fill them (for example with the median).

In [ ]:
print("Total null values:", int(df.isnull().sum().sum()))
df.isnull().sum()

## 2.7 Encoding the Categorical Columns

`store_primary_category` has 73 different values. As in the problem brief, we convert the column to the `category` type and take its integer codes. `market_id` and `order_protocol` are stored as decimals (`1.0`, `2.0`), so we convert them to integers.

In [ ]:
df["store_category_code"] = df["store_primary_category"].astype("category").cat.codes
df["market_id"] = df["market_id"].astype(int)
df["order_protocol"] = df["order_protocol"].astype(int)

df[["store_primary_category", "store_category_code"]].drop_duplicates().sort_values("store_category_code").head()

In [ ]:
df = df.drop(columns=["store_primary_category"])

df.head()

**Note:** `market_id` and `order_protocol` are also labels, so we turn them into one-hot columns (`pd.get_dummies`) just before modelling in Section 4. That keeps the original columns available for the charts in Section 3.

In [ ]:
print("Shape after Section 2:", df.shape)
df.dtypes

<a id="sec3"></a>

# Section 3: Exploratory Visualisation and Outlier Treatment (20 points)

Now that the data is clean and complete, we look at it with charts (countplots and scatterplots, as in the brief), check for outliers, remove them, and plot the data again to see whether anything improved.

## 3.1 Visualising the Data

We import `IsolationForest`, which we use in Section 3.4, and list the eight numeric columns that describe the order and the partner supply.

In [ ]:
from sklearn.ensemble import IsolationForest

num_cols = ["total_items", "subtotal", "num_distinct_items", "min_item_price", "max_item_price",
            "total_onshift_dashers", "total_busy_dashers", "total_outstanding_orders"]

day_names = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

### Countplots of the Categorical Columns

A countplot shows how many orders fall into each category.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

sns.countplot(data=df, x="market_id", color=BLUE, ax=axes[0])
axes[0].bar_label(axes[0].containers[0], fmt="{:,.0f}", fontsize=8)
axes[0].set_title("Orders per market")
axes[0].set_ylabel("number of orders")

sns.countplot(data=df, x="order_protocol", color=BLUE, ax=axes[1])
axes[1].bar_label(axes[1].containers[0], fmt="{:,.0f}", fontsize=8)
axes[1].set_title("Orders per order protocol")
axes[1].set_ylabel("")

top_codes = df["store_category_code"].value_counts().head(10).index
sns.countplot(data=df, y="store_category_code", order=top_codes, color=BLUE, ax=axes[2])
axes[2].bar_label(axes[2].containers[0], fmt="{:,.0f}", fontsize=8)
axes[2].set_title("10 most common restaurant categories")
axes[2].set_xlabel("number of orders")

plt.tight_layout()
plt.show()

### Scatterplots (Before Removing Outliers)

We plot delivery time against subtotal, and total items against subtotal. We define the plots as a function, so that exactly the same plot can be drawn again after the outliers are removed. A random sample of 20000 orders keeps the plot readable.

In [ ]:
def scatter_pair(data, title):
    sample = data.sample(20000, random_state=42)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sns.scatterplot(data=sample, x="subtotal", y="time_taken_mins", s=8, alpha=0.2, color=BLUE, edgecolor=None, ax=axes[0])
    axes[0].set_title("Delivery time vs subtotal")
    sns.scatterplot(data=sample, x="subtotal", y="total_items", s=8, alpha=0.2, color=BLUE, edgecolor=None, ax=axes[1])
    axes[1].set_title("Total items vs subtotal")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

scatter_pair(df, "Before removing outliers")

### Box Plots to Check for Outliers (Before Removing Outliers)

A box plot shows the middle half of the values as a box and marks the unusual values as dots. Each column has its own scale.

In [ ]:
def box_grid(data, title):
    fig, axes = plt.subplots(2, 4, figsize=(16, 6))
    for ax, col in zip(axes.ravel(), num_cols):
        sns.boxplot(x=data[col], color=BLUE, fliersize=2, flierprops={"markerfacecolor": ORANGE, "markeredgecolor": ORANGE}, ax=ax)
        ax.set_title(col, fontsize=10)
        ax.set_xlabel("")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

box_grid(df, "Before removing outliers")

## 3.2 Why Check for Outliers? (Brief Question 4)

An **outlier** is a value that sits far away from the rest of the data. We check for them before training because:

- **Averages and spreads get pulled.** One order with 411 items moves the mean and the standard deviation of `total_items`, and `StandardScaler` uses both.
- **Squared errors reward big values.** The mean squared error squares every mistake, so a few extreme rows can dominate the loss and drag the weights of a neural network towards them.
- **Models learn from too little evidence.** A very rare kind of order has so few examples that the model cannot learn it, but it can still distort the rest.
- **They may be mistakes.** Some outliers are typing or system errors that should not be learned at all.

**Note:** An outlier is not automatically wrong. A very large order can be real. So we look at the outliers first, and then decide what to do with them.

## 3.3 Three Methods to Find and Remove Outliers (Brief Question 5)

| Method | How it works | Looks at |
|:--|:--|:--|
| **IQR rule** (the rule behind a box plot) | A value is an outlier if it is more than 1.5 times the IQR (the width of the middle half) below the first quartile or above the third quartile | One column at a time |
| **Z-score** | A value is an outlier if it is more than 3 standard deviations from the mean | One column at a time |
| **Isolation Forest** (also Local Outlier Factor, DBSCAN) | Random trees try to isolate each row. Rows that are isolated with very few splits are unusual | Several columns at once |

The brief allows any method. We use **Isolation Forest**, because it looks at several columns together instead of one column in isolation.

## 3.4 Detect and Remove Outliers

`contamination=0.01` tells the Isolation Forest to flag about 1% of the rows as the most unusual ones. We give it only the eight numeric input columns and **not** the delivery time, so the target does not decide which rows are removed.

In [ ]:
df_before = df.copy()

iso = IsolationForest(n_estimators=100, contamination=0.01, random_state=42, n_jobs=-1)
iso.fit(df[num_cols])
df["is_outlier"] = iso.predict(df[num_cols]) == -1

print("Rows flagged as outliers:", df["is_outlier"].sum(), f"({df['is_outlier'].mean() * 100:.2f}% of the data)")

Let's see what the flagged rows look like. In the charts they are drawn in orange.

In [ ]:
sample = df.sample(30000, random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.scatterplot(data=sample, x="subtotal", y="total_items", hue="is_outlier", palette={False: BLUE, True: ORANGE},
                s=10, alpha=0.5, edgecolor=None, ax=axes[0])
axes[0].set_title("Flagged rows: total items vs subtotal")
sns.scatterplot(data=sample, x="total_onshift_dashers", y="total_outstanding_orders", hue="is_outlier", palette={False: BLUE, True: ORANGE},
                s=10, alpha=0.5, edgecolor=None, ax=axes[1])
axes[1].set_title("Flagged rows: orders waiting vs partners on shift")
plt.tight_layout()
plt.show()

In [ ]:
df.groupby("is_outlier")[num_cols + ["time_taken_mins"]].mean().T.round(1)

Now we remove the flagged rows.

In [ ]:
df = df[~df["is_outlier"]].drop(columns="is_outlier").reset_index(drop=True)

print("Rows before:", len(df_before))
print("Rows after :", len(df))

## 3.5 Compare the Plots and Results Before and After

We draw exactly the same scatterplots and box plots again and compare them with the earlier ones.

In [ ]:
scatter_pair(df, "After removing outliers")

In [ ]:
box_grid(df, "After removing outliers")

The numbers tell the same story:

In [ ]:
def summary(data):
    return pd.Series({
        "rows": len(data),
        "max total_items": data["total_items"].max(),
        "max subtotal": data["subtotal"].max(),
        "max total_outstanding_orders": data["total_outstanding_orders"].max(),
        "std of subtotal": data["subtotal"].std(),
        "std of delivery time": data["time_taken_mins"].std(),
        "corr(time, subtotal)": data["time_taken_mins"].corr(data["subtotal"]),
    })

pd.DataFrame({"before": summary(df_before), "after": summary(df)}).round(2)

## 3.6 Order Volume and Delivery Time by Day and Hour

Does the time of the order change how long delivery takes? The charts on the left count the orders, and the charts on the right show the average delivery time with its 95% confidence band.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 9))

sns.countplot(data=df, x="day_of_week", color=BLUE, ax=axes[0, 0])
axes[0, 0].set_xticks(range(7))
axes[0, 0].set_xticklabels(day_names)
axes[0, 0].set_title("Orders by day of the week")
axes[0, 0].set_xlabel("")

sns.barplot(data=df, x="day_of_week", y="time_taken_mins", color=BLUE, errorbar=("ci", 95), n_boot=200, ax=axes[0, 1])
axes[0, 1].set_xticks(range(7))
axes[0, 1].set_xticklabels(day_names)
axes[0, 1].set_ylim(35, 55)
axes[0, 1].set_title("Average delivery time by day of the week")
axes[0, 1].set_xlabel("")
axes[0, 1].set_ylabel("minutes")

sns.countplot(data=df, x="hour", color=BLUE, ax=axes[1, 0])
axes[1, 0].set_title("Orders by hour of the day")

sns.lineplot(data=df, x="hour", y="time_taken_mins", marker="o", color=BLUE, errorbar=("ci", 95), n_boot=200, ax=axes[1, 1])
axes[1, 1].set_title("Average delivery time by hour of the day")
axes[1, 1].set_ylabel("minutes")

plt.tight_layout()
plt.show()

In [ ]:
print("Average delivery time by hour (minutes):")
print(df.groupby("hour")["time_taken_mins"].mean().round(1).to_string())
print("\nAverage delivery time by day of the week (minutes):")
print(df.groupby("day_of_week")["time_taken_mins"].mean().round(1).rename(dict(enumerate(day_names))).to_string())

<a id="sec4"></a>

# Section 4: Baseline Model: Random Forest (10 points)

Before building a neural network we need a baseline: a strong, simple model whose numbers the neural network must beat. As in the brief, we split the data into train and test sets, train a Random Forest, and check its MSE, RMSE and MAE.

## 4.1 Preparing the Features and Splitting the Data

`market_id` and `order_protocol` are labels, so we turn them into one-hot columns with `pd.get_dummies(..., dtype=float)`. `store_category_code` keeps its integer codes. Then `X` holds the inputs and `y` the target.

In [ ]:
X = pd.get_dummies(df.drop(columns=["time_taken_mins"]), columns=["market_id", "order_protocol"], dtype=float)
y = df["time_taken_mins"]

print(X.shape, y.shape)
X.head()

`train_test_split` keeps 20% of the rows for testing. `random_state=42` makes the split reproducible, so every run gives the same train and test sets. We do not use `stratify`, because the target is a continuous number.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Train:", X_train.shape, y_train.shape)
print("Test :", X_test.shape, y_test.shape)

The test set must look like the training set. Let's check the delivery time in both.

In [ ]:
split_target = pd.concat([
    pd.DataFrame({"minutes": y_train, "set": "train"}),
    pd.DataFrame({"minutes": y_test, "set": "test"}),
])

plt.figure(figsize=(9, 4))
sns.histplot(data=split_target, x="minutes", hue="set", binwidth=2, stat="density", common_norm=False,
             element="step", fill=False, palette=[BLUE, ORANGE])
plt.title("Delivery time in the train and test sets")
plt.show()

## 4.2 Random Forest Regression

`RandomForestRegressor` builds many decision trees on random parts of the data and averages their predictions. We use 100 trees, `n_jobs=-1` to use all processor cores, and a fixed `random_state`.

**Note:** A Random Forest splits on thresholds, so it does not need scaled features (see Section 5.1).

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

## 4.3 Evaluate the Random Forest

We report MSE, RMSE and MAE, as in the brief, and also R² (the share of the variation in delivery time that the model explains, which tells us how accurate the model is). MAE is the average miss in minutes, RMSE is also in minutes but punishes big misses more, and MSE is the quantity a neural network minimises. The training scores next to the test scores show whether the forest has memorised the training data.

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def regression_report(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return dict(MSE=mse, RMSE=mse ** 0.5, MAE=mean_absolute_error(y_true, y_pred), R2=r2_score(y_true, y_pred))

rf_train_pred = rf.predict(X_train)
rf_test_pred = rf.predict(X_test)

rf_train_res = regression_report(y_train, rf_train_pred)
rf_test_res = regression_report(y_test, rf_test_pred)

pd.DataFrame({"Random Forest (train)": rf_train_res, "Random Forest (test)": rf_test_res}).round(3)

Two charts show where the Random Forest is right and wrong. On the left, each dot is an order: a perfect model would put every dot on the dashed diagonal. On the right is the distribution of the errors (actual minus predicted).

In [ ]:
test_sample = pd.DataFrame({"actual": y_test.values, "predicted": rf_test_pred}).sample(8000, random_state=42)
test_sample["error"] = test_sample["actual"] - test_sample["predicted"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.scatterplot(data=test_sample, x="actual", y="predicted", s=8, alpha=0.3, color=ORANGE, edgecolor=None, ax=axes[0])
axes[0].plot([30, 110], [30, 110], color="#52514e", linestyle="--", linewidth=1)
axes[0].set_title("Random Forest: predicted vs actual (test)")
axes[0].set_xlabel("actual minutes")
axes[0].set_ylabel("predicted minutes")

sns.histplot(test_sample["error"], bins=60, kde=True, color=ORANGE, ax=axes[1])
axes[1].axvline(0, color="#52514e", linestyle="--", linewidth=1)
axes[1].set_title("Random Forest: errors (actual - predicted)")
axes[1].set_xlabel("minutes")
plt.tight_layout()
plt.show()

## 4.4 Other Classical Methods for Regression (Brief Question 6)

Besides Random Forest, these classical machine learning methods can predict a number:

| Method | Idea |
|:--|:--|
| Linear Regression (and Ridge, Lasso) | A weighted sum of the features, with optional penalties on large weights |
| Decision Tree Regressor | One tree of yes/no splits |
| Gradient Boosting (XGBoost, LightGBM, `GradientBoostingRegressor`) | Many small trees, each correcting the errors of the previous ones |
| K-Nearest Neighbours Regressor | The average of the most similar training orders |
| Support Vector Regression | A kernel method that fits a tube around the data |

<a id="sec5"></a>

# Section 5: Neural Network Regression and Model Comparison (30 points)

Now we build a neural network in PyTorch, following the structure used in the course: scale the data, build the model, choose the loss and the optimizer, write our own training loop, and try different configurations one at a time. At the end we compare the network with the Random Forest.

## 5.1 Scaling the Data (Brief Question 7)

Neural networks learn with gradient descent, so the size of the numbers matters. `subtotal` runs into thousands while `hour` is between 0 and 23. Unscaled inputs give a stretched, lopsided loss surface, and gradient descent zig-zags across it. A forest does not care, because it only compares each feature with a threshold.

**Standardisation** (`z = (x - mean) / std`) puts every column on the same footing. We fit the scaler on the **training set only** and just transform the other sets, otherwise information from data the model must never see would leak into training.

We also scale the **target**. A network starts with outputs near 0 while delivery times are around 46, so scaling `y` makes the first steps of training sensible. Predictions are converted back to minutes with `inverse_transform` before we calculate any metric.

To plot the training and validation loss we need a validation set. We set aside 12.5% of the training rows for it, so the network trains on 70% of the data, validates on 10%, and the test set (20%) stays untouched.

In [ ]:
from sklearn.preprocessing import StandardScaler

X_train_nn, X_val, y_train_nn, y_val = train_test_split(X_train, y_train, test_size=0.125, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train_nn)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)

y_scaler = StandardScaler()
y_train_s = y_scaler.fit_transform(y_train_nn.values.reshape(-1, 1))
y_val_s = y_scaler.transform(y_val.values.reshape(-1, 1))

y_std = y_scaler.scale_[0]
print("Train / validation / test rows:", len(X_train_nn), len(X_val), len(X_test))
print("Target mean and std used for scaling:", round(y_scaler.mean_[0], 2), round(y_std, 2))

The box plots show the effect: on the left the raw columns live on completely different scales, on the right they are comparable.

In [ ]:
show_cols = ["total_items", "subtotal", "max_item_price", "total_onshift_dashers", "total_outstanding_orders", "hour"]
scaled_df = pd.DataFrame(X_train_s, columns=X.columns)

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
sns.boxplot(data=X_train_nn[show_cols], orient="h", color=BLUE, fliersize=1, ax=axes[0])
axes[0].set_title("Before scaling")
sns.boxplot(data=scaled_df[show_cols], orient="h", color=BLUE, fliersize=1, ax=axes[1])
axes[1].set_title("After scaling (mean 0, std 1)")
axes[1].set_yticklabels([])
plt.tight_layout()
plt.show()

PyTorch models work with tensors. Features become `float32`. The target is a column of shape `(N, 1)`, which is the shape of the model output. A `DataLoader` hands the model small shuffled batches. `seed_everything` fixes NumPy's and PyTorch's random numbers so that runs can be compared fairly.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

def seed_everything(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)

seed_everything()

X_train_t = torch.tensor(X_train_s, dtype=torch.float32)
X_val_t = torch.tensor(X_val_s, dtype=torch.float32)
X_test_t = torch.tensor(X_test_s, dtype=torch.float32)

y_train_t = torch.tensor(y_train_s, dtype=torch.float32)
y_val_t = torch.tensor(y_val_s, dtype=torch.float32)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=256, shuffle=True)

xb, yb = next(iter(train_loader))
print("PyTorch", torch.__version__)
print("One batch -> features:", xb.shape, " targets:", yb.shape)

## 5.2 Defining the Network Architecture and the Activation Functions (Brief Question 9)

The baseline network has two hidden layers:

| Layer | Description |
|:--|:--|
| Input | One neuron for each of the 24 input columns |
| Hidden 1 | 128 neurons with ReLU |
| Hidden 2 | 64 neurons with ReLU |
| Output | **1 neuron with no activation** |

**Which activation functions, and why?**

- **Hidden layers: ReLU**, `max(0, z)`. It is cheap, it keeps a gradient of exactly 1 for positive inputs so gradients do not vanish as they do with sigmoid and tanh, and it is the default in most modern networks. Its weakness is that a unit stuck at negative values stops learning (dying ReLU), which Leaky ReLU fixes. We test the alternatives in Section 5.5.
- **Output layer: linear (no activation).** The last neuron decides what the network predicts. A linear last neuron gives regression, and sigmoid or softmax would give classification. A sigmoid would squeeze the answer into 0 to 1, and a ReLU would cut off negative values, which the scaled target can take.

The chart compares the four activation functions we try, and their derivatives (the gradient signal that flows back during training).

In [ ]:
z = torch.linspace(-4, 4, 400, requires_grad=True)
activations = {"ReLU": torch.relu, "Leaky ReLU": lambda t: nn.functional.leaky_relu(t, 0.01), "Tanh": torch.tanh, "Sigmoid": torch.sigmoid}

curves = []
for name, f in activations.items():
    out = f(z)
    grad, = torch.autograd.grad(out.sum(), z)
    curves.append(pd.DataFrame({"z": z.detach().numpy(), "value": out.detach().numpy(), "derivative": grad.numpy(), "activation": name}))
curves = pd.concat(curves)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
sns.lineplot(data=curves, x="z", y="value", hue="activation", palette=[BLUE, ORANGE, AQUA, YELLOW], ax=axes[0])
axes[0].set_title("Activation functions")
sns.lineplot(data=curves, x="z", y="derivative", hue="activation", palette=[BLUE, ORANGE, AQUA, YELLOW], legend=False, ax=axes[1])
axes[1].set_title("Their derivatives")
plt.tight_layout()
plt.show()

`make_net` builds the network from a list of hidden sizes and an activation. As in the course, weights start with Xavier initialisation and biases at zero, and the last layer has **no** activation.

In [ ]:
def make_net(hidden=(128, 64), act=nn.ReLU):
    layers, d = [], X_train_t.shape[1]
    for w in hidden:
        layers.append(nn.Linear(d, w))
        layers.append(act())
        d = w
    model = nn.Sequential(*layers, nn.Linear(d, 1))        # last layer: no activation
    for layer in model:
        if isinstance(layer, nn.Linear):
            nn.init.xavier_uniform_(layer.weight)
            nn.init.zeros_(layer.bias)
    return model

torch.manual_seed(0)
model = make_net()
print(model)
print("Trainable parameters:", sum(p.numel() for p in model.parameters()))

## 5.3 Loss Function and Optimizer (Brief Question 8)

- **Loss: `nn.MSELoss()`**, the mean squared error. It is the standard loss for a continuous target, it is smooth, and it punishes big misses more than small ones. We report results as RMSE in minutes, which is easier to read.
- **Optimizer: Adam**, with learning rate 0.001 to start. Plain gradient descent uses one fixed learning rate for every weight and zig-zags on a lopsided loss surface. Momentum smooths the direction, RMSprop shrinks the step where gradients have been large, and **Adam combines both**, with a correction for its first steps. It is a strong default that needs little tuning. Adam still needs a learning rate, so we compare learning rates in Section 5.5.

In [ ]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# one forward and backward pass on one batch, before any training
loss = criterion(model(xb), yb)
loss.backward()
print("Loss before training (scaled units):", round(loss.item(), 3))
print("Gradient shapes -> first layer:", tuple(model[0].weight.grad.shape), " last layer:", tuple(model[-1].weight.grad.shape))

## 5.4 Training the Network and Plotting the Losses

We write the training loop ourselves, as in the course. For every batch: `zero_grad`, forward pass, loss, `backward`, `step`. At the end of every epoch we switch to `model.eval()`, measure the error on the training and validation sets with `torch.no_grad()`, and record it in `h`. Errors are recorded as RMSE in minutes. After training, the function loads the weights of the epoch with the lowest validation error.

In [ ]:
def fit(model, opt, epochs=40, bs=256):
    loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=bs, shuffle=True)
    h = {"train": [], "val": []}
    best, best_state = float("inf"), None
    for ep in range(epochs):
        model.train()
        for xb, yb in loader:
            opt.zero_grad(); criterion(model(xb), yb).backward(); opt.step()
        model.eval()
        with torch.no_grad():
            h["train"].append(criterion(model(X_train_t), y_train_t).item() ** 0.5 * y_std)
            v = criterion(model(X_val_t), y_val_t).item()
            h["val"].append(v ** 0.5 * y_std)
        if v < best:
            best, best_state = v, {k: t.clone() for k, t in model.state_dict().items()}
    model.load_state_dict(best_state)
    h["best_epoch"] = int(np.argmin(h["val"]))
    return h

def show(h, title=""):
    curves = pd.DataFrame({"epoch": range(1, len(h["train"]) + 1), "training": h["train"], "validation": h["val"]})
    curves = curves.melt(id_vars="epoch", var_name="set", value_name="RMSE (minutes)")
    plt.figure(figsize=(9, 4.5))
    sns.lineplot(data=curves, x="epoch", y="RMSE (minutes)", hue="set", palette=[BLUE, ORANGE])
    plt.axvline(h["best_epoch"] + 1, color="#52514e", linestyle=":", linewidth=1.5, label="best epoch")
    plt.title(title)
    plt.legend()
    plt.show()

Let's train the baseline for 60 epochs and plot the training and validation loss.

In [ ]:
torch.manual_seed(0)
base = make_net()
h_base = fit(base, torch.optim.Adam(base.parameters(), lr=1e-3), epochs=60)
show(h_base, "Baseline network: training and validation loss")

In [ ]:
print("Last epoch -> train RMSE", round(h_base["train"][-1], 3), "| validation RMSE", round(h_base["val"][-1], 3), "minutes")
print("Best epoch ->", h_base["best_epoch"] + 1, "with validation RMSE", round(h_base["val"][h_base["best_epoch"]], 3), "minutes")

## 5.5 Trying Different Configurations

We now follow the debugging method from the Practical Aspects class: **change one thing at a time** and keep a scoreboard. Every run starts from the same seed, so differences come from the change and not from luck. Each run trains for 40 epochs and keeps the weights of the best epoch. `run` records the training error, the last validation error, the best validation error and the best epoch.

In [ ]:
results = {}
histories = {}
configs = {}

def make_optimizer(params, optimizer, lr):
    if optimizer == "adam":
        return torch.optim.Adam(params, lr=lr)
    if optimizer == "rmsprop":
        return torch.optim.RMSprop(params, lr=lr)
    if optimizer == "sgd":
        return torch.optim.SGD(params, lr=lr)
    return torch.optim.SGD(params, lr=lr, momentum=0.9)       # "sgd_momentum"

def run(name, group, hidden=(128, 64), act=nn.ReLU, optimizer="adam", lr=1e-3, bs=256, epochs=40):
    torch.manual_seed(0)
    m = make_net(hidden, act)
    opt = make_optimizer(m.parameters(), optimizer, lr)
    h = fit(m, opt, epochs=epochs, bs=bs)
    histories[name] = h
    configs[name] = dict(hidden=hidden, act=act, optimizer=optimizer, lr=lr, bs=bs)
    results[name] = [group, h["train"][-1], h["val"][-1], min(h["val"]), h["best_epoch"] + 1]

def scoreboard(group):
    table = pd.DataFrame(results, index=["group", "train_RMSE", "val_RMSE", "best_val_RMSE", "best_epoch"]).T
    table = table[table["group"] == group].drop(columns="group")
    return table.astype(float).round(3)

def plot_group(group, title):
    table = scoreboard(group).sort_values("best_val_RMSE")
    plt.figure(figsize=(9, 0.6 * len(table) + 1.5))
    ax = sns.barplot(x=table["best_val_RMSE"], y=table.index, color=BLUE)
    ax.bar_label(ax.containers[0], fmt="%.3f", fontsize=9)
    plt.title(title)
    plt.xlabel("best validation RMSE (minutes)")
    plt.ylabel("")
    plt.show()

def plot_curves(names, title, palette):
    long = pd.concat([pd.DataFrame({"epoch": range(1, len(histories[n]["val"]) + 1), "validation RMSE": histories[n]["val"], "run": n}) for n in names])
    plt.figure(figsize=(9, 4.5))
    sns.lineplot(data=long, x="epoch", y="validation RMSE", hue="run", palette=palette, linewidth=2)
    plt.title(title)
    plt.show()

### Experiment 1: Size of the Network

Wider and deeper networks can learn more, but also need more data and time.

In [ ]:
for hidden in [(32,), (64, 32), (128, 64), (256, 128, 64), (128, 64, 32, 16)]:
    run(f"size {hidden}", "size", hidden=hidden)

scoreboard("size")

In [ ]:
plot_group("size", "Experiment 1: size of the network")

### Experiment 2: Activation Function

Same network, only the hidden activation changes.

In [ ]:
for name, act in [("ReLU", nn.ReLU), ("Leaky ReLU", nn.LeakyReLU), ("Tanh", nn.Tanh), ("Sigmoid", nn.Sigmoid)]:
    run(f"activation {name}", "activation", act=act)

scoreboard("activation")

In [ ]:
plot_group("activation", "Experiment 2: activation function")

### Experiment 3: The Optimiser Race

The same network with four optimisers: plain SGD and SGD with momentum (learning rate 0.01), RMSprop and Adam (learning rate 0.001), as in the SwiftLend case.

In [ ]:
run("optimiser SGD", "optimiser", optimizer="sgd", lr=0.01)
run("optimiser SGD + Momentum", "optimiser", optimizer="sgd_momentum", lr=0.01)
run("optimiser RMSprop", "optimiser", optimizer="rmsprop", lr=1e-3)
run("optimiser Adam", "optimiser", optimizer="adam", lr=1e-3)

scoreboard("optimiser")

In [ ]:
plot_curves(["optimiser SGD", "optimiser SGD + Momentum", "optimiser RMSprop", "optimiser Adam"],
            "Experiment 3: validation RMSE for four optimisers", [GRAY, YELLOW, AQUA, BLUE])

### Experiment 4: Learning Rate

The learning rate is the first hyperparameter to tune. As in the Practical Aspects class we try values that differ by large factors, a log-scale spread, with Adam.

In [ ]:
for lr in [1e-4, 3e-4, 1e-3, 3e-3, 1e-2]:
    run(f"learning rate {lr:g}", "learning rate", lr=lr)

scoreboard("learning rate")

In [ ]:
plot_curves([f"learning rate {lr:g}" for lr in [1e-4, 3e-4, 1e-3, 3e-3, 1e-2]],
            "Experiment 4: validation RMSE for five learning rates", [GRAY, YELLOW, AQUA, BLUE, ORANGE])

### Experiment 5: Batch Size

A small batch updates the weights more often but with noisier gradients. A large batch gives smoother gradients and a faster epoch, but fewer updates.

In [ ]:
for bs in [64, 128, 256, 512, 1024]:
    run(f"batch size {bs}", "batch size", bs=bs)

scoreboard("batch size")

In [ ]:
plot_group("batch size", "Experiment 5: batch size")

### The Full Scoreboard

Every run from the five experiments in one chart, coloured by experiment.

In [ ]:
board = pd.DataFrame(results, index=["group", "train_RMSE", "val_RMSE", "best_val_RMSE", "best_epoch"]).T
board[["train_RMSE", "val_RMSE", "best_val_RMSE", "best_epoch"]] = board[["train_RMSE", "val_RMSE", "best_val_RMSE", "best_epoch"]].astype(float)
board = board.sort_values("best_val_RMSE")

plt.figure(figsize=(10, 9))
sns.barplot(data=board, x="best_val_RMSE", y=board.index, hue="group", dodge=False,
            palette=[BLUE, ORANGE, AQUA, YELLOW, GRAY])
plt.title("All experiments: best validation RMSE (lower is better)")
plt.xlabel("minutes")
plt.ylabel("")
plt.legend(title="experiment")
plt.show()

board.head(5).round(3)

## 5.6 The Final Model

We take the configuration with the lowest validation error on the scoreboard and train it for 60 epochs. The weights of the best epoch are kept, and the loss curves are plotted. The test set is not used for any of these choices.

In [ ]:
best_name = board.index[0]
best_cfg = configs[best_name]
print("Best configuration:", best_name)
print(best_cfg)

In [ ]:
torch.manual_seed(0)
final = make_net(best_cfg["hidden"], best_cfg["act"])
final_opt = make_optimizer(final.parameters(), best_cfg["optimizer"], best_cfg["lr"])

h_final = fit(final, final_opt, epochs=60, bs=best_cfg["bs"])
print("Best epoch:", h_final["best_epoch"] + 1, "| validation RMSE:", round(min(h_final["val"]), 3), "minutes")
show(h_final, "Final model: training and validation loss")

## 5.7 Final Test Evaluation and Comparison with the Random Forest

Now, once, we use the test set. The network predicts in scaled units, so `y_scaler.inverse_transform` converts the predictions back to minutes. Both models are scored with exactly the same metrics on exactly the same test rows.

In [ ]:
final.eval()
with torch.no_grad():
    nn_test_pred = y_scaler.inverse_transform(final(X_test_t).numpy()).ravel()

nn_test_res = regression_report(y_test, nn_test_pred)

comparison = pd.DataFrame({"Random Forest": rf_test_res, "Neural Network": nn_test_res})
comparison.round(3)

The chart compares the two models on the three error metrics of the brief.

In [ ]:
bars = comparison.loc[["MSE", "RMSE", "MAE"]].reset_index().melt(id_vars="index", var_name="model", value_name="value")

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
for ax, metric in zip(axes, ["MSE", "RMSE", "MAE"]):
    sns.barplot(data=bars[bars["index"] == metric], x="model", y="value", hue="model",
                palette={"Random Forest": ORANGE, "Neural Network": BLUE}, legend=False, ax=ax)
    ax.bar_label(ax.containers[0], fmt="%.2f", fontsize=10)
    ax.bar_label(ax.containers[1], fmt="%.2f", fontsize=10)
    ax.set_title(f"{metric} (lower is better)")
    ax.set_xlabel("")
    ax.set_ylabel("")
plt.tight_layout()
plt.show()

Predicted against actual for each model, with the dashed diagonal as the perfect prediction.

In [ ]:
pred_frame = pd.DataFrame({"actual": y_test.values, "Random Forest": rf_test_pred, "Neural Network": nn_test_pred})
sample = pred_frame.sample(8000, random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), sharex=True, sharey=True)
for ax, name, color in zip(axes, ["Random Forest", "Neural Network"], [ORANGE, BLUE]):
    sns.scatterplot(data=sample, x="actual", y=name, s=8, alpha=0.3, color=color, edgecolor=None, ax=ax)
    ax.plot([30, 110], [30, 110], color="#52514e", linestyle="--", linewidth=1)
    ax.set_title(f"{name}: predicted vs actual (test)")
    ax.set_xlabel("actual minutes")
    ax.set_ylabel("predicted minutes")
plt.tight_layout()
plt.show()

The distribution of the errors (actual minus predicted) for both models.

In [ ]:
errors = pd.DataFrame({"Random Forest": y_test.values - rf_test_pred, "Neural Network": y_test.values - nn_test_pred})
errors_long = errors.melt(var_name="model", value_name="error (minutes)")

plt.figure(figsize=(9, 4.5))
sns.kdeplot(data=errors_long, x="error (minutes)", hue="model", palette={"Random Forest": ORANGE, "Neural Network": BLUE},
            fill=True, alpha=0.25, common_norm=False, linewidth=2)
plt.axvline(0, color="#52514e", linestyle="--", linewidth=1)
plt.title("Test set: distribution of errors")
plt.show()

## 5.8 Why Neural Networks Perform Well on Large Datasets (Brief Question 10)

A neural network has many parameters (ours has about 13,000), and every parameter has to be estimated from data. With little data there are too few examples to pin them all down, so the network overfits or does not train well. With a lot of data, the same flexibility becomes an advantage: the network can learn complicated, smooth relationships and build its own features, and it keeps improving as more examples arrive. Many classical models flatten out earlier, because their capacity is limited by the features we hand them. Our dataset has more than 170,000 deliveries, which is large enough for the network to learn well without overfitting, as the training and validation curves in Section 5.4 show.